# NER-Enhanced Resume Classification
This notebook demonstrates how to use extracted entities to enhance resume classification accuracy.

## Features:
- Uses extracted entities as additional features
- Combines text classification with entity-based classification
- Enhanced feature engineering for better accuracy


In [1]:
import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')


In [3]:
!pip install spacy
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 5.7 MB/s eta 0:00:03
     -------- ------------------------------- 2.6/12.8 MB 8.9 MB/s eta 0:00:02
     --------------- ------------------------ 5.0/12.8 MB 10.4 MB/s eta 0:00:01
     ------------------- -------------------- 6.3/12.8 MB 9.4 MB/s eta 0:00:01
     ---------------------- ----------------- 7.3/12.8 MB 8.4 MB/s eta 0:00:01
     ------------------------- -------------- 8.1/12.8 MB 7.4 MB/s eta 0:00:01
     --------------------------- ------------ 8.7/12.8 MB 6.3 MB/s eta 0:00:01
     ---------------------------- ----------- 9.2/12.8 MB 5.8 MB/s eta 0:00:01
     ----------------------------- ---------- 9.4/12.8 MB 5.5 MB/s eta 0:00:01
     ------------------------------- -------- 10.0/12.8 MB 4.8 MB/s eta 0:00:01
     ------------------------------- -------- 10.2/12.8 MB 4.7 MB/s eta 0:00:01
     --------------------------------- ------ 10.7/12.8 


[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Load Data with NER Features


In [6]:
# Load processed data with NER features
train = pd.read_parquet("../data/processed/classification_train.parquet")
val = pd.read_parquet("../data/processed/classification_val.parquet")
test = pd.read_parquet("../data/processed/classification_test.parquet")

print("Training data shape:", train.shape)
print("Validation data shape:", val.shape)
print("Test data shape:", test.shape)

# Check for NER features
entity_columns = [col for col in train.columns if col.startswith('entity_')]
print(f"\nNER entity columns found: {len(entity_columns)}")
print("Entity columns:", entity_columns)

if entity_columns:
    print("\nSample entity features:")
    for col in entity_columns[:3]:
        print(f"{col}: {train[col].iloc[0][:100]}..." if len(str(train[col].iloc[0])) > 100 else f"{col}: {train[col].iloc[0]}")
else:
    print("\n⚠️ No entity columns found. Make sure NER is enabled in config.yaml and preprocessing was run.")


Training data shape: (9372, 2)
Validation data shape: (1339, 2)
Test data shape: (2678, 2)

NER entity columns found: 0
Entity columns: []

⚠️ No entity columns found. Make sure NER is enabled in config.yaml and preprocessing was run.


## Enhanced Feature Engineering


In [7]:
def create_enhanced_features(df, entity_columns):
    """Create enhanced features combining text and entities"""
    df_enhanced = df.copy()
    
    # Combine text with entity features
    if entity_columns:
        # Create combined text feature
        entity_texts = []
        for _, row in df.iterrows():
            entity_parts = []
            for col in entity_columns:
                if pd.notna(row[col]) and row[col]:
                    entity_parts.append(str(row[col]))
            entity_texts.append(" | ".join(entity_parts))
        
        df_enhanced['entity_text'] = entity_texts
        
        # Create combined feature for ML models
        df_enhanced['combined_text'] = df_enhanced['text'] + " | " + df_enhanced['entity_text']
        
        # Create entity count features
        for col in entity_columns:
            df_enhanced[f'{col}_count'] = df_enhanced[col].apply(
                lambda x: len(str(x).split(' | ')) if pd.notna(x) and x else 0
            )
    else:
        df_enhanced['entity_text'] = ""
        df_enhanced['combined_text'] = df_enhanced['text']
    
    return df_enhanced

# Create enhanced features
train_enhanced = create_enhanced_features(train, entity_columns)
val_enhanced = create_enhanced_features(val, entity_columns)
test_enhanced = create_enhanced_features(test, entity_columns)

print("Enhanced training data shape:", train_enhanced.shape)
print("New columns:", [col for col in train_enhanced.columns if col not in train.columns])


Enhanced training data shape: (9372, 4)
New columns: ['entity_text', 'combined_text']


## Traditional ML Approach with Entity Features


In [8]:
# Prepare data for traditional ML
le = LabelEncoder()
train_enhanced['label_id'] = le.fit_transform(train_enhanced['label'])
val_enhanced['label_id'] = le.transform(val_enhanced['label'])
test_enhanced['label_id'] = le.transform(test_enhanced['label'])

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_enhanced['combined_text'])
X_val = vectorizer.transform(val_enhanced['combined_text'])
X_test = vectorizer.transform(test_enhanced['combined_text'])

y_train = train_enhanced['label_id']
y_val = val_enhanced['label_id']
y_test = test_enhanced['label_id']

print(f"TF-IDF features shape: {X_train.shape}")
print(f"Number of classes: {len(le.classes_)}")


TF-IDF features shape: (9372, 5000)
Number of classes: 43


In [9]:
# Train Random Forest Classifier
print("Training Random Forest Classifier...")
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_classifier.fit(X_train, y_train)

# Evaluate on validation set
rf_val_pred = rf_classifier.predict(X_val)
rf_val_acc = accuracy_score(y_val, rf_val_pred)

# Evaluate on test set
rf_test_pred = rf_classifier.predict(X_test)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

print(f"Random Forest Validation Accuracy: {rf_val_acc:.4f}")
print(f"Random Forest Test Accuracy: {rf_test_acc:.4f}")

# Feature importance (top 20)
feature_names = vectorizer.get_feature_names_out()
importances = rf_classifier.feature_importances_
top_features = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)[:20]

print("\nTop 20 Important Features:")
for feature, importance in top_features:
    print(f"{feature}: {importance:.4f}")


Training Random Forest Classifier...
Random Forest Validation Accuracy: 0.8260
Random Forest Test Accuracy: 0.8234

Top 20 Important Features:
operations manager: 0.0075
advocate: 0.0073
apparel: 0.0069
civil: 0.0065
aviation: 0.0064
pmo: 0.0058
sap: 0.0057
electrical: 0.0057
mechanical: 0.0054
construction: 0.0053
testing: 0.0053
business analyst: 0.0052
accountant: 0.0050
consultant: 0.0049
fitness: 0.0048
sales: 0.0047
banking: 0.0044
financial: 0.0044
engineer: 0.0042
mechanical engineer: 0.0042


In [10]:
# Train Logistic Regression
print("Training Logistic Regression...")
lr_classifier = LogisticRegression(random_state=42, max_iter=1000, n_jobs=-1)
lr_classifier.fit(X_train, y_train)

# Evaluate on validation set
lr_val_pred = lr_classifier.predict(X_val)
lr_val_acc = accuracy_score(y_val, lr_val_pred)

# Evaluate on test set
lr_test_pred = lr_classifier.predict(X_test)
lr_test_acc = accuracy_score(y_test, lr_test_pred)

print(f"Logistic Regression Validation Accuracy: {lr_val_acc:.4f}")
print(f"Logistic Regression Test Accuracy: {lr_test_acc:.4f}")


Training Logistic Regression...
Logistic Regression Validation Accuracy: 0.7886
Logistic Regression Test Accuracy: 0.8077


## Transformer Model with Entity Features


In [11]:
# Use combined text for transformer model
model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Prepare datasets
ds = DatasetDict({
    "train": Dataset.from_pandas(train_enhanced[['combined_text', 'label_id']].rename(columns={'combined_text': 'text'})),
    "validation": Dataset.from_pandas(val_enhanced[['combined_text', 'label_id']].rename(columns={'combined_text': 'text'})),
    "test": Dataset.from_pandas(test_enhanced[['combined_text', 'label_id']].rename(columns={'combined_text': 'text'})),
})

# Tokenize
MAX_LEN = 256
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

ds_tok = ds.map(tokenize, batched=True, desc="Tokenizing")
ds_tok = ds_tok.rename_column("label_id", "labels")
ds_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("Dataset prepared for transformer training")


Tokenizing:   0%|          | 0/9372 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1339 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/2678 [00:00<?, ? examples/s]

Dataset prepared for transformer training


In [16]:
# Ensure LabelEncoder is fitted before using
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(train_enhanced["label"])  # or whatever your label column is named
print("Label encoder fitted. Classes:", list(le.classes_))


Label encoder fitted. Classes: ['Accountant', 'Advocate', 'Agriculture', 'Apparel', 'Architecture', 'Arts', 'Automobile', 'Aviation', 'BPO', 'Banking', 'Blockchain', 'Building and Construction', 'Business Analyst', 'Civil Engineer', 'Consultant', 'Data Science', 'Database', 'Designing', 'DevOps', 'Digital Media', 'DotNet Developer', 'ETL Developer', 'Education', 'Electrical Engineering', 'Finance', 'Food and Beverages', 'Health and Fitness', 'Human Resources', 'Information Technology', 'Java Developer', 'Management', 'Mechanical Engineer', 'Network Security Engineer', 'Operations Manager', 'PMO', 'Public Relations', 'Python Developer', 'React Developer', 'SAP Developer', 'SQL Developer', 'Sales', 'Testing', 'Web Designing']


In [17]:
# ============================================================
# Model setup and training with optimized runtime (same flow)
# ============================================================

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Model setup
num_labels = len(le.classes_)  # from your label encoder earlier
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Training arguments
args = TrainingArguments(
    output_dir="artifacts/ner_enhanced_distilbert",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ✅ Optimizations (no change in flow)
    fp16=True,                   # enables mixed-precision training for faster GPU compute
    dataloader_num_workers=4,    # speeds up data loading
    save_total_limit=2,          # keeps last two best checkpoints only
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    tokenizer=tokenizer,
)

print("\n🚀 Starting transformer fine-tuning with enhanced NER features...\n")

# Start training
train_output = trainer.train()

print("\n✅ Training completed! Best model loaded.\n")

# Save the final model
trainer.save_model("artifacts/ner_enhanced_distilbert/final_model")

# Evaluate final model on validation set
metrics = trainer.evaluate()
print("\n📊 Final Evaluation Metrics:", metrics)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 Starting transformer fine-tuning with enhanced NER features...



  0%|          | 0/3516 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(ds_tok["test"])
print(f"NER-Enhanced Transformer Test Results:")
for key, value in test_results.items():
    print(f"{key}: {value:.4f}")


## Model Comparison


In [ ]:
# Compare all models
results = {
    'Model': ['Random Forest', 'Logistic Regression', 'NER-Enhanced Transformer'],
    'Test Accuracy': [rf_test_acc, lr_test_acc, test_results.get('eval_accuracy', 0.0)],
    'Validation Accuracy': [rf_val_acc, lr_val_acc, test_results.get('eval_loss', 0.0)]
}

results_df = pd.DataFrame(results)
print("\nModel Comparison:")
print(results_df.to_string(index=False))

# Find best model
best_model_idx = results_df['Test Accuracy'].idxmax()
best_model = results_df.loc[best_model_idx, 'Model']
best_accuracy = results_df.loc[best_model_idx, 'Test Accuracy']

print(f"\n🏆 Best Model: {best_model} with {best_accuracy:.4f} accuracy")


## Entity Analysis


In [ ]:
if entity_columns:
    # Analyze entity distribution
    print("Entity Feature Analysis:")
    print("=" * 50)
    
    for col in entity_columns:
        non_empty = train_enhanced[col].apply(lambda x: pd.notna(x) and str(x).strip() != '')
        coverage = non_empty.mean() * 100
        avg_length = train_enhanced[col].apply(lambda x: len(str(x).split(' | ')) if pd.notna(x) and x else 0).mean()
        
        print(f"{col}:")
        print(f"  Coverage: {coverage:.1f}%")
        print(f"  Avg entities per resume: {avg_length:.1f}")
        
        # Show sample entities
        sample_entities = train_enhanced[col].dropna().head(3).tolist()
        for i, entity in enumerate(sample_entities, 1):
            print(f"  Sample {i}: {entity[:100]}{'...' if len(str(entity)) > 100 else ''}")
        print()
else:
    print("No entity features found. Make sure NER is enabled in config.yaml and preprocessing was run.")


## Save Enhanced Model


In [ ]:
# Save the best model
if best_model == 'NER-Enhanced Transformer':
    model.save_pretrained("artifacts/ner_enhanced_distilbert_final")
    tokenizer.save_pretrained("artifacts/ner_enhanced_distilbert_final")
    print("✅ NER-enhanced transformer model saved")
elif best_model == 'Random Forest':
    import joblib
    joblib.dump(rf_classifier, "artifacts/ner_enhanced_rf_model.pkl")
    joblib.dump(vectorizer, "artifacts/ner_enhanced_vectorizer.pkl")
    joblib.dump(le, "artifacts/ner_enhanced_label_encoder.pkl")
    print("✅ NER-enhanced Random Forest model saved")
elif best_model == 'Logistic Regression':
    import joblib
    joblib.dump(lr_classifier, "artifacts/ner_enhanced_lr_model.pkl")
    joblib.dump(vectorizer, "artifacts/ner_enhanced_vectorizer.pkl")
    joblib.dump(le, "artifacts/ner_enhanced_label_encoder.pkl")
    print("✅ NER-enhanced Logistic Regression model saved")

print("\n🎉 NER-enhanced training completed!")
